# Práctica 1: Predicción del precio de productos

**Ingeniería del Conocimiento · Grado en Ingeniería Informática**

| | |
|---|---|
| **Alumno/a** | NOMBRE APELLIDOS |
| **URL del vídeo (YouTube, oculto)** | https://youtu.be/XXXXXXXXXXX |
| **Grupo** | Jose, Irene, Alberto, Pablo, Juan |
| **Repositorio** | https://github.com/Liria18/NOMBRE-REPO |

> Cada alumno entrega su propia copia de este informe con su nombre y la URL de su vídeo en esta portada.

## 1. Resumen del trabajo

El objetivo es predecir `price` de productos de e-commerce a partir de 14 variables (7 numéricas y 7 categóricas). La métrica es RMSE (a partir de 57 se obtiene un 5).

Se ha hecho un EDA, un preprocesado con variables nuevas, una comparación de 6 modelos con validación cruzada de 5 folds, un ajuste de hiperparámetros y un modelo final que mezcla Ridge y LightGBM sobre `log(price)`. El RMSE en validación cruzada del modelo final es de aproximadamente **52.4**.

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete src/ desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.config import DATA_RAW, TARGET, CATEGORICAL, NUMERIC
sns.set_theme(style="whitegrid")
from src.preprocessing import add_features, split_xy
from src.models import evaluar_cv
from src.modelo import build_final_model

## 2. Principales resultados del análisis exploratorio

*(Ver `01_eda.ipynb`)*

- 8000 productos en train y 2000 en test.
- Solo `material` tiene valores perdidos (~3%).
- `price` es muy asimétrico (cola hasta 1238 €); `log(price)` es casi simétrico.
- `weight_kg` es la variable más relacionada con el precio, seguida de `shipping_cost` y `margin`. `discount` e `is_promo` lo reducen.
- `brand_tier` es ordinal y cada marca tiene un único tier.

In [ ]:
train = pd.read_csv(DATA_RAW / "train.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train[TARGET], bins=60, ax=ax[0]).set_title("price")
sns.histplot(np.log(train[TARGET]), bins=60, ax=ax[1]).set_title("log(price)")
plt.show()

## 3. Preprocesamiento aplicado

*(Ver `02_preprocesado.ipynb` y `src/preprocessing.py`)*

- Imputación de `material` con la categoría `Desconocido` e indicador `material_missing`.
- Variables nuevas: logaritmos de peso, reviews y envío, `tier_num`, factores multiplicativos de margen y descuento, `envio_por_kg`.
- One-hot + escalado para modelos lineales; codificación ordinal para árboles.
- Todo dentro de un `Pipeline` para evitar fuga de información en la CV.

## 4. Modelos y rendimiento en validación

*(Ver `03_modelos.ipynb`, `04_tuning.ipynb` y `src/modelo.py`)*

RMSE en CV de 5 folds (KFold, barajado, semilla 42):

| Modelo | RMSE CV |
|---|---|
| RandomForest | 55.84 |
| HistGradientBoosting | 54.33 |
| LinearRegression | 53.58 |
| Ridge | 53.58 |
| LightGBM (log price) | 53.10 |
| Ridge (log price) | 52.45 |
| **Final: 0.7 Ridge log (alpha 0.3) + 0.3 LightGBM log ajustado** | **52.38** |

El modelo final entrena sobre `log(price)` y corrige el sesgo al volver a euros con el factor de *smearing* de Duan.

In [ ]:
X, y = split_xy(train)
X = add_features(X)
por_fold, oof = evaluar_cv(build_final_model(), X, y)
print(f"RMSE CV modelo final: {por_fold.mean():.3f} ± {por_fold.std():.3f}")
plt.scatter(oof, y, s=5, alpha=0.3); plt.plot([0, 1200], [0, 1200], "r--")
plt.xlabel("predicción"); plt.ylabel("precio real"); plt.show()

## 5. Generación de la submission

`python -m src.train NombreAlumno` genera `submissions/submission_NombreAlumno.csv` con 2000 filas y columnas `id,prediction`.

## 6. Dificultades encontradas y mejoras futuras

**Dificultades**
- Unos pocos productos muy caros concentran gran parte del RMSE y son difíciles de anticipar.
- Al deshacer el logaritmo se infravaloran los precios; se resolvió con la corrección de smearing.
- Coordinar a 5 personas con Git (ramas, PRs, conflictos en notebooks).

**Mejoras futuras**
- Probar más modelos (CatBoost, XGBoost) y stacking.
- Búsqueda de hiperparámetros más amplia (Optuna).
- Modelar explícitamente los precios extremos.